# Quantum Reservoir Computing System — Quickstart

Created by School of AI and School of QC.

This notebook runs a compact, leakage-safe benchmark: a fixed quantum reservoir, a size-matched echo-state network, two lag-based models, and persistence. It then inspects one-step forecasts, recursive behavior, memory, finite-shot sensitivity, and saved-model inference.

## 1. Imports

Install the project from the repository root with `python -m pip install -e .` before running the notebook.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

from quantum_reservoir_system.config import ExperimentConfig
from quantum_reservoir_system.experiment import run_experiment
from quantum_reservoir_system.inference import forecast_frame

## 2. Configure a compact reproducible run

The full reference configuration uses four qubits and 1,600 points. This smaller configuration is intended for an interactive walkthrough. The chronological split, train-only scaling, and validation-based readout selection remain unchanged.

In [ ]:
config = ExperimentConfig(
    sample_count=600,
    burn_in=100,
    qubits=3,
    virtual_nodes=2,
    washout=20,
    lag_count=10,
    rollout_horizon=16,
    random_forest_estimators=80,
    shot_counts=(128, 512),
    output_root="artifacts/notebook",
).validate()
config

## 3. Run the complete experiment

Qiskit constructs the Hamiltonian and Pauli observables. SciPy performs exact full-Hamiltonian density-matrix evolution. Only the ridge readout is trained for the QRC.

In [ ]:
result = run_experiment(config, progress_callback=print)
print(f"Artifacts: {result.output_directory}")

## 4. Compare held-out one-step metrics

The lowest RMSE wins this narrow task. Also inspect inference time, directional accuracy, and skill against persistence. A one-step result is not evidence of stable recursive forecasting.

In [ ]:
metric_columns = [
    "model",
    "mae",
    "rmse",
    "r2",
    "directional_accuracy",
    "skill_vs_persistence_rmse",
]
result.metrics[metric_columns]

In [ ]:
display(Image(filename=str(result.output_directory / "one_step_forecasts.png")))
display(Image(filename=str(result.output_directory / "recursive_rollout.png")))

## 5. Inspect reservoir diagnostics

The trace and unitarity checks should remain close to floating-point precision. The saved circuit is a first-order visualization of a virtual step; it is not the exact simulation path.

In [ ]:
result.diagnostics["quantum"]

In [ ]:
capacity = (
    result.memory_capacity.groupby("reservoir")["positive_capacity"]
    .sum()
    .sort_values(ascending=False)
)
display(capacity)
display(result.shot_sensitivity[["shots", "rmse", "r2"]])
display(Image(filename=str(result.output_directory / "memory_capacity.png")))
display(Image(filename=str(result.output_directory / "shot_sensitivity.png")))

## 6. Reload the saved models for recursive inference

Saved inference reconstructs the exact reservoir and reloads fitted classical models. Load joblib artifacts only from sources you trust.

In [ ]:
history = result.series.tail(config.lag_count)[["value"]]
future = forecast_frame(result.output_directory, history, horizon=8)
future

In [ ]:
forecast_columns = [column for column in future.columns if column != "horizon_step"]
future.set_index("horizon_step")[forecast_columns].plot(
    figsize=(10, 4), title="Saved-model recursive forecasts"
)
plt.ylabel("Forecast value")
plt.show()

## Interpretation checklist

- Compare QRC with the strongest classical result, not only persistence.
- Keep one-step and recursive results separate.
- Repeat seeds and chronological origins before drawing research conclusions.
- Treat independent finite-shot perturbation as a sensitivity probe, not hardware emulation.
- Do not claim quantum speedup or production readiness from this simulation.

Created by School of AI and School of QC.